# dropout_bench

Reordered working notebook derived from `main_markdown_revised.ipynb`.

This version only **reorders existing cells** and inserts a few **placeholder cells** for future refactoring. Existing P-cells keep their original content; new planning cells use section letters **A–E**.

# A. Foundation

This section establishes the benchmark foundation.

It covers:
- imports, configuration, and reproducibility
- dataset download and availability checks
- output reset and directory initialization
- DuckDB setup and source registration
- raw structural audit
- enrollment backbone construction
- survival-ready event and time construction

These steps create the canonical data and storage foundation used by all later sections.


## A0 - Dependency Check and Auto-Install

Run this step before A1 to ensure required Python libraries are available in the active environment.

In [1]:
# ==============================================================
# A0 - Dependency Check and Auto-Install
# --------------------------------------------------------------
# Purpose:
#   Validate required third-party libraries and install missing
#   packages in the current Python environment before A1 imports.
# ==============================================================

import importlib.util
import subprocess
import sys

print("\n" + "=" * 70)
print("A0 - Dependency Check and Auto-Install")
print("=" * 70)

# ------------------------------
# 1) Required and optional packages
# ------------------------------
REQUIRED_PACKAGES = {
    "duckdb": "duckdb",
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "sklearn": "scikit-learn",
    "lifelines": "lifelines",
    "torch": "torch",
}

OPTIONAL_PACKAGES = {
    "torchtuples": "torchtuples",
    "pycox": "pycox",
}



A0 - Dependency Check and Auto-Install


### Funcao: is_available

Definicao isolada de is_available para reutilizacao nas etapas seguintes.


In [2]:


def is_available(module_name: str) -> bool:
    return importlib.util.find_spec(module_name) is not None


### Funcao: install_package

Definicao isolada de install_package para reutilizacao nas etapas seguintes.


In [3]:


def install_package(package_name: str) -> tuple[bool, str]:
    cmd = [sys.executable, "-m", "pip", "install", package_name]
    result = subprocess.run(cmd, capture_output=True, text=True)
    ok = result.returncode == 0
    details = result.stdout.strip() if ok else result.stderr.strip()
    return ok, details


In [4]:


# ------------------------------
# 2) Install missing required packages
# ------------------------------
required_missing = [m for m in REQUIRED_PACKAGES if not is_available(m)]
optional_missing = [m for m in OPTIONAL_PACKAGES if not is_available(m)]

if required_missing:
    print("Missing required modules:", ", ".join(required_missing))
else:
    print("All required modules are already installed.")

install_failures = []

for module_name in required_missing:
    package_name = REQUIRED_PACKAGES[module_name]
    print(f"Installing required package: {package_name}")
    ok, details = install_package(package_name)
    if not ok:
        install_failures.append((module_name, package_name, details))

# ------------------------------
# 3) Try to install optional packages
# ------------------------------
if optional_missing:
    print("Missing optional modules:", ", ".join(optional_missing))

for module_name in optional_missing:
    package_name = OPTIONAL_PACKAGES[module_name]
    print(f"Installing optional package: {package_name}")
    ok, details = install_package(package_name)
    if not ok:
        print(f"Optional package failed to install: {package_name}")
        print(details.splitlines()[-1] if details else "No error details available.")

# ------------------------------
# 4) Final validation
# ------------------------------
still_missing_required = [m for m in REQUIRED_PACKAGES if not is_available(m)]

if still_missing_required:
    print("\nFailed required modules:", ", ".join(still_missing_required))
    if install_failures:
        print("\nInstall errors (required):")
        for module_name, package_name, details in install_failures:
            print(f"- {module_name} ({package_name})")
            print(details.splitlines()[-1] if details else "No error details available.")
    raise RuntimeError(
        "A0 could not install all required dependencies. "
        "Fix the errors above and rerun A0 before A1."
    )

print("\nDependency check completed. You can now run A1 safely.")


All required modules are already installed.

Dependency check completed. You can now run A1 safely.


## A1 — Imports, Configuration, and Reproducibility


In [5]:
# ==============================================================
# A1 — Imports, Configuration, and Reproducibility
# --------------------------------------------------------------
# Purpose:
#   Centralize imports, global configuration, random seeds,
#   project paths, output directories, and lightweight helpers
#   for reproducible execution.
#
# Outputs:
#   - Global configuration variables
#   - Reproducibility seeds
#   - Standard output directory tree
#   - Environment/version summary
# ==============================================================

# ------------------------------
# 1) Standard library
# ------------------------------
from pathlib import Path
from datetime import datetime
import os
import json
import random
import warnings
import platform
import requests
import zipfile
import duckdb
import shutil

# ------------------------------
# 2) Core scientific stack
# ------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------
# 3) Machine learning / survival
# ------------------------------
from sklearn import __version__ as sklearn_version
from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from lifelines import CoxPHFitter, CoxTimeVaryingFitter
import lifelines

# ------------------------------
# 4) Deep learning
# ------------------------------
import torch
import torch.nn as nn

# Optional: enable only if installed / needed later
try:
    import torchtuples as tt
    TORCHTUPLES_AVAILABLE = True
except ImportError:
    TORCHTUPLES_AVAILABLE = False

try:
    from pycox.models import LogisticHazard, CoxPH
    PYCOX_AVAILABLE = True
    PYCOX_IMPORT_ERROR = None
except Exception as e:
    PYCOX_AVAILABLE = False
    PYCOX_IMPORT_ERROR = str(e)

# ------------------------------
# 5) Global display / warning settings
# ------------------------------
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)
pd.set_option("display.max_rows", 200)

# ------------------------------
# 6) Reproducibility
# ------------------------------
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

# More deterministic behavior (may reduce speed a bit)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ------------------------------
# 7) Project paths
# ------------------------------
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "content"          # same convention you already use
OUTPUT_DIR = PROJECT_ROOT / "outputs_benchmark_survival"

TABLES_DIR = OUTPUT_DIR / "tables"
FIGURES_DIR = OUTPUT_DIR / "figures"
MODELS_DIR = OUTPUT_DIR / "models"
METADATA_DIR = OUTPUT_DIR / "metadata"

for d in [OUTPUT_DIR, TABLES_DIR, FIGURES_DIR, MODELS_DIR, METADATA_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ------------------------------
# 8) Protocol configuration
# ------------------------------
CONFIG = {
    "seed": SEED,
    "project_root": str(PROJECT_ROOT),
    "data_dir": str(DATA_DIR),
    "output_dir": str(OUTPUT_DIR),
    "tables_dir": str(TABLES_DIR),
    "figures_dir": str(FIGURES_DIR),
    "models_dir": str(MODELS_DIR),
    "metadata_dir": str(METADATA_DIR),
    "unit_of_analysis": "enrollment",
    "enrollment_key": ["id_student", "code_module", "code_presentation"],
    "time_granularity": "weekly",
    "event_definition": "Withdrawn with valid date_unregistration",
    "test_size": 0.30,
    "temporal_buckets_q": 4,
}


### Funcao: print_section

Definicao isolada de print_section para reutilizacao nas etapas seguintes.


In [6]:

# ------------------------------
# 9) Lightweight utility helpers
# ------------------------------
def print_section(title: str) -> None:
    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)


### Funcao: save_json

Definicao isolada de save_json para reutilizacao nas etapas seguintes.


In [7]:

def save_json(obj: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


### Funcao: get_environment_summary

Definicao isolada de get_environment_summary para reutilizacao nas etapas seguintes.


In [8]:

def get_environment_summary() -> dict:
    return {
        "timestamp": datetime.now().isoformat(),
        "python_version": platform.python_version(),
        "platform": platform.platform(),
        "pandas_version": pd.__version__,
        "numpy_version": np.__version__,
        "sklearn_version": sklearn_version,
        "lifelines_version": lifelines.__version__,
        "torch_version": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
        "torchtuples_available": TORCHTUPLES_AVAILABLE,
        "pycox_available": PYCOX_AVAILABLE,
        "seed": SEED,
    }


In [9]:
# ------------------------------
# 10) Persist metadata and shared TOML configuration
# ------------------------------
env_summary = get_environment_summary()
save_json(CONFIG, METADATA_DIR / "config.json")
save_json(env_summary, METADATA_DIR / "environment_summary.json")


def _toml_value(v):
    if isinstance(v, bool):
        return "true" if v else "false"
    if isinstance(v, int):
        return str(v)
    if isinstance(v, float):
        return repr(v)
    if isinstance(v, str):
        escaped = v.replace("\\", "\\\\").replace('"', '\\"')
        return f'"{escaped}"'
    if isinstance(v, (list, tuple)):
        return "[" + ", ".join(_toml_value(x) for x in v) + "]"
    raise TypeError(f"Unsupported TOML value type: {type(v)}")


def save_toml_sections(obj: dict, path: Path) -> None:
    lines = []
    for section, values in obj.items():
        lines.append(f"[{section}]")
        for key, value in values.items():
            lines.append(f"{key} = {_toml_value(value)}")
        lines.append("")
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text("\n".join(lines).strip() + "\n", encoding="utf-8")


shared_config = {
    "paths": {
        "data_dir": "content",
        "output_dir": "outputs_benchmark_survival",
        "tables_subdir": "tables",
        "figures_subdir": "figures",
        "models_subdir": "models",
        "metadata_subdir": "metadata",
        "data_output_subdir": "data",
        "duckdb_filename": "benchmark_survival.duckdb",
    },
    "benchmark": {
        "seed": SEED,
        "test_size": 0.30,
        "temporal_buckets_q": CONFIG.get("temporal_buckets_q", 4),
        "early_window_weeks": 4,
        "main_enrollment_window_weeks": 4,
        "unit_of_analysis": "enrollment",
        "time_granularity": "weekly",
        "event_definition": "Withdrawn with valid date_unregistration",
    },
    "keys": {
        "enrollment_key": ["id_student", "code_module", "code_presentation"],
    },
}

CONFIG_TOML_PATH = PROJECT_ROOT / "benchmark_shared_config.toml"
save_toml_sections(shared_config, CONFIG_TOML_PATH)
save_json(shared_config, METADATA_DIR / "benchmark_shared_config.json")

# ------------------------------
# 11) Execution summary
# ------------------------------
print_section("A1 — Imports, Configuration, and Reproducibility")
print("PROJECT_ROOT :", PROJECT_ROOT)
print("DATA_DIR     :", DATA_DIR)
print("OUTPUT_DIR   :", OUTPUT_DIR)
print("SEED         :", SEED)
print("CUDA         :", torch.cuda.is_available())
print("PYCOX        :", PYCOX_AVAILABLE)
print("TORCHTUPLES  :", TORCHTUPLES_AVAILABLE)

if PYCOX_IMPORT_ERROR is not None:
    print("PYCOX ERROR  :", PYCOX_IMPORT_ERROR)

print("\nSaved metadata files:")
print("-", METADATA_DIR / "config.json")
print("-", METADATA_DIR / "environment_summary.json")
print("-", METADATA_DIR / "benchmark_shared_config.json")
print("\nSaved shared TOML:")
print("-", CONFIG_TOML_PATH)


A1 — Imports, Configuration, and Reproducibility
PROJECT_ROOT : /workspaces/survival_dropout_benchmark
DATA_DIR     : /workspaces/survival_dropout_benchmark/content
OUTPUT_DIR   : /workspaces/survival_dropout_benchmark/outputs_benchmark_survival
SEED         : 42
CUDA         : False
PYCOX        : True
TORCHTUPLES  : True

Saved metadata files:
- /workspaces/survival_dropout_benchmark/outputs_benchmark_survival/metadata/config.json
- /workspaces/survival_dropout_benchmark/outputs_benchmark_survival/metadata/environment_summary.json
- /workspaces/survival_dropout_benchmark/outputs_benchmark_survival/metadata/benchmark_shared_config.json

Saved shared TOML:
- /workspaces/survival_dropout_benchmark/benchmark_shared_config.toml


### A1.1 — Download and Extract OULAD Dataset

In [10]:
# ==============================================================
# A1.1 — Download and Extract OULAD Dataset
# --------------------------------------------------------------
# Purpose:
#   Ensure the OULAD raw files are available under DATA_DIR.
#   If needed, download the official zip file and extract it
#   into the project content directory.
#
# Notes:
#   - Reuses the project paths defined in A1
#   - Avoids re-downloading if the zip already exists
#   - Avoids re-extracting if the required CSV files already exist
# ==============================================================

import requests
import zipfile

print_section("A1.1 — Download and Extract OULAD Dataset")

# ------------------------------
# 1) Dataset source and paths
# ------------------------------
BASE_PATH = Path.cwd()
DATASET_URL = "https://analyse.kmi.open.ac.uk/open-dataset/download"
ZIP_FILENAME = "anonymisedData.zip"
ZIP_PATH = PROJECT_ROOT / ZIP_FILENAME

EXPECTED_CSVS = [
    "studentInfo.csv",
    "studentRegistration.csv",
    "studentVle.csv",
    "courses.csv",
    "vle.csv",
    "studentAssessment.csv",
    "assessments.csv",
]



A1.1 — Download and Extract OULAD Dataset


### Funcao: list_missing_csvs

Definicao isolada de list_missing_csvs para reutilizacao nas etapas seguintes.


In [11]:

# ------------------------------
# 2) Helper checks
# ------------------------------
def list_missing_csvs(data_dir: Path, expected_files: list[str]) -> list[str]:
    return [fname for fname in expected_files if not (data_dir / fname).exists()]


### Funcao: dataset_is_ready

Definicao isolada de dataset_is_ready para reutilizacao nas etapas seguintes.


In [12]:

def dataset_is_ready(data_dir: Path, expected_files: list[str]) -> bool:
    return len(list_missing_csvs(data_dir, expected_files)) == 0


### Funcao: download_oulad_zip

Definicao isolada de download_oulad_zip para reutilizacao nas etapas seguintes.


In [13]:

# ------------------------------
# 3) Download function
# ------------------------------
def download_oulad_zip(url: str, zip_path: Path) -> None:
    if zip_path.exists():
        print(f"Zip file already exists: {zip_path}")
        return

    print(f"Downloading OULAD zip from: {url}")
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/122.0.0.0 Safari/537.36"
        )
    }

    try:
        with requests.get(url, headers=headers, stream=True, timeout=60) as response:
            response.raise_for_status()
            with open(zip_path, "wb") as f:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
        print(f"Download completed: {zip_path}")
    except Exception as e:
        if zip_path.exists():
            zip_path.unlink()
        raise RuntimeError(f"Download failed: {e}")


### Funcao: extract_oulad_zip

Definicao isolada de extract_oulad_zip para reutilizacao nas etapas seguintes.


In [14]:

# ------------------------------
# 4) Extraction function
# ------------------------------
def extract_oulad_zip(zip_path: Path, extract_dir: Path) -> None:
    if not zip_path.exists():
        raise FileNotFoundError(f"Zip file not found: {zip_path}")

    extract_dir.mkdir(parents=True, exist_ok=True)

    print(f"Extracting zip into: {extract_dir}")
    try:
        with zipfile.ZipFile(zip_path, "r") as zip_ref:
            zip_ref.extractall(extract_dir)
        print("Extraction completed successfully.")
    except zipfile.BadZipFile:
        raise RuntimeError(f"Bad zip file: {zip_path}")
    except Exception as e:
        raise RuntimeError(f"Extraction failed: {e}")


In [15]:

# ------------------------------
# 5) Main execution logic
# ------------------------------
DATA_DIR.mkdir(parents=True, exist_ok=True)

missing_before = list_missing_csvs(DATA_DIR, EXPECTED_CSVS)

if dataset_is_ready(DATA_DIR, EXPECTED_CSVS):
    print("OULAD dataset is already available in DATA_DIR. Skipping download and extraction.")
else:
    print("Missing files before setup:")
    for fname in missing_before:
        print(f" - {fname}")

    download_oulad_zip(DATASET_URL, ZIP_PATH)
    extract_oulad_zip(ZIP_PATH, DATA_DIR)

# ------------------------------
# 6) Final validation
# ------------------------------
missing_after = list_missing_csvs(DATA_DIR, EXPECTED_CSVS)

print("\nFinal validation:")
print(f"DATA_DIR: {DATA_DIR}")

if missing_after:
    print("Some expected CSV files are still missing after extraction:")
    for fname in missing_after:
        print(f" - {fname}")
    raise FileNotFoundError("Dataset extraction finished, but required OULAD files are missing.")
else:
    print("All expected OULAD CSV files are available.")
    for fname in EXPECTED_CSVS:
        print(f" - {(DATA_DIR / fname).resolve()}")

# ------------------------------
# 7) Save dataset metadata
# ------------------------------
dataset_metadata = {
    "dataset_url": DATASET_URL,
    "zip_filename": ZIP_FILENAME,
    "zip_path": str(ZIP_PATH),
    "data_dir": str(DATA_DIR),
    "expected_csvs": EXPECTED_CSVS,
    "missing_before": missing_before,
    "missing_after": missing_after,
}

save_json(dataset_metadata, METADATA_DIR / "dataset_setup.json")
print("\nSaved:")
print("-", (METADATA_DIR / "dataset_setup.json").resolve())


OULAD dataset is already available in DATA_DIR. Skipping download and extraction.

Final validation:
DATA_DIR: /workspaces/survival_dropout_benchmark/content
All expected OULAD CSV files are available.
 - /workspaces/survival_dropout_benchmark/content/studentInfo.csv
 - /workspaces/survival_dropout_benchmark/content/studentRegistration.csv
 - /workspaces/survival_dropout_benchmark/content/studentVle.csv
 - /workspaces/survival_dropout_benchmark/content/courses.csv
 - /workspaces/survival_dropout_benchmark/content/vle.csv
 - /workspaces/survival_dropout_benchmark/content/studentAssessment.csv
 - /workspaces/survival_dropout_benchmark/content/assessments.csv

Saved:
- /workspaces/survival_dropout_benchmark/outputs_benchmark_survival/metadata/dataset_setup.json


### A1.2 — Reset Output Directory

In [16]:
# ==============================================================
# A1.2 — Reset Output Directory
# --------------------------------------------------------------
# Purpose:
#   Remove the previous benchmark output directory so the notebook
#   can recreate a clean output structure for the current run.
# ==============================================================

print("\n" + "=" * 70)
print("A1.2 — Reset Output Directory")
print("=" * 70)

from pathlib import Path
import shutil

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = ["OUTPUT_DIR"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run the required upstream cells first."
    )

target_path = Path(OUTPUT_DIR)

# ------------------------------
# 2) Reset output directory
# ------------------------------
if target_path.exists():
    shutil.rmtree(target_path)
    print(f"Directory '{target_path.name}' successfully deleted at: {target_path}")
else:
    print(f"Directory '{target_path.name}' does not exist. Nothing to delete.")


A1.2 — Reset Output Directory
Directory 'outputs_benchmark_survival' successfully deleted at: /workspaces/survival_dropout_benchmark/outputs_benchmark_survival


### A1.3 — Create Required Output Directory Tree

In [17]:
# ==============================================================
# A1.3 — Create Required Output Directory Tree
# --------------------------------------------------------------
# Purpose:
#   Recreate the canonical output directory tree required by the
#   benchmark after the output reset step.
# ==============================================================
from pathlib import Path

# ------------------------------
# 1) Basic checks (reuse upstream variable)
# ------------------------------
required_names = ["OUTPUT_DIR"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run A1 first."
    )

OUTPUT_DIR = Path(OUTPUT_DIR)

print("\n" + "=" * 70)
print("A1.3 — Create Required Output Directory Tree")
print("=" * 70)

REQUIRED_DIRS = [
    OUTPUT_DIR,  # outputs_benchmark_survival
    OUTPUT_DIR / "data",
    OUTPUT_DIR / "tables",
    OUTPUT_DIR / "tables" / "paper_main",
    OUTPUT_DIR / "tables" / "paper_appendix",
    OUTPUT_DIR / "figures",
    OUTPUT_DIR / "models",
    OUTPUT_DIR / "metadata",
]

for d in REQUIRED_DIRS:
    d.mkdir(parents=True, exist_ok=True)

print("Directory tree ensured:")
for d in REQUIRED_DIRS:
    print(f" - {d}")


A1.3 — Create Required Output Directory Tree
Directory tree ensured:
 - /workspaces/survival_dropout_benchmark/outputs_benchmark_survival
 - /workspaces/survival_dropout_benchmark/outputs_benchmark_survival/data
 - /workspaces/survival_dropout_benchmark/outputs_benchmark_survival/tables
 - /workspaces/survival_dropout_benchmark/outputs_benchmark_survival/tables/paper_main
 - /workspaces/survival_dropout_benchmark/outputs_benchmark_survival/tables/paper_appendix
 - /workspaces/survival_dropout_benchmark/outputs_benchmark_survival/figures
 - /workspaces/survival_dropout_benchmark/outputs_benchmark_survival/models
 - /workspaces/survival_dropout_benchmark/outputs_benchmark_survival/metadata


## A2 — DuckDB Setup and OULAD Registration


In [18]:
# ==============================================================
# A2 — DuckDB Setup and OULAD Registration
# --------------------------------------------------------------
# Purpose:
#   Initialize a persistent DuckDB database for the project,
#   register the extracted OULAD CSV files as SQL views,
#   validate access to each view, and save an initial registry
#   audit for downstream data preparation steps.
#
# Inputs expected from previous cells:
#   - PROJECT_ROOT
#   - DATA_DIR
#   - OUTPUT_DIR
#   - TABLES_DIR
#   - METADATA_DIR
#   - save_json
#   - print_section
#
# Outputs:
#   - benchmark_survival.duckdb
#   - table_duckdb_registered_views.csv
#   - duckdb_setup.json
# ==============================================================

# ------------------------------
# 1) Imports
# ------------------------------
print_section("A2 — DuckDB Setup and OULAD Registration")

# ------------------------------
# 2) Paths and directories
# ------------------------------
DATA_OUTPUT_DIR = OUTPUT_DIR / "data"
DATA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DUCKDB_PATH = OUTPUT_DIR / "benchmark_survival.duckdb"

EXPECTED_FILES = {
    "studentInfo": DATA_DIR / "studentInfo.csv",
    "studentRegistration": DATA_DIR / "studentRegistration.csv",
    "studentVle": DATA_DIR / "studentVle.csv",
    "courses": DATA_DIR / "courses.csv",
    "vle": DATA_DIR / "vle.csv",
    "studentAssessment": DATA_DIR / "studentAssessment.csv",
    "assessments": DATA_DIR / "assessments.csv",
}

# ------------------------------
# 3) Validate expected CSV files
# ------------------------------
missing_files = [name for name, path in EXPECTED_FILES.items() if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        f"Missing expected OULAD file(s) in DATA_DIR={DATA_DIR}:\n- " + "\n- ".join(missing_files)
    )

print(f"DuckDB file will be created/used at: {DUCKDB_PATH}")
print("All expected OULAD CSV files are available.")

# ------------------------------
# 4) Open persistent DuckDB connection
# ------------------------------
con = duckdb.connect(str(DUCKDB_PATH))

# ------------------------------
# 5) Register CSVs as views
# ------------------------------
registered_views = []

for view_name, csv_path in EXPECTED_FILES.items():
    sql = f"""
    CREATE OR REPLACE VIEW {view_name} AS
    SELECT *
    FROM read_csv_auto('{csv_path.as_posix()}', HEADER=TRUE);
    """
    con.execute(sql)
    registered_views.append(view_name)

print("\nRegistered DuckDB views:")
for v in registered_views:
    print(f" - {v}")

# ------------------------------
# 6) Validate row counts and columns
# ------------------------------
audit_rows = []

for view_name in registered_views:
    row_count = con.execute(f"SELECT COUNT(*) FROM {view_name}").fetchone()[0]

    columns_info = con.execute(f"DESCRIBE {view_name}").fetchdf()
    n_cols = columns_info.shape[0]
    col_names = columns_info["column_name"].tolist()

    audit_rows.append({
        "view_name": view_name,
        "n_rows": int(row_count),
        "n_cols": int(n_cols),
        "columns": ", ".join(col_names),
        "source_csv": str(EXPECTED_FILES[view_name])
    })

audit_duckdb_views = pd.DataFrame(audit_rows).sort_values("view_name").reset_index(drop=True)

# ------------------------------
# 7) Save audit outputs
# ------------------------------
audit_csv_path = TABLES_DIR / "table_duckdb_registered_views.csv"
audit_duckdb_views.to_csv(audit_csv_path, index=False)

duckdb_setup_metadata = {
    "duckdb_path": str(DUCKDB_PATH),
    "data_output_dir": str(DATA_OUTPUT_DIR),
    "registered_views": registered_views,
    "source_files": {k: str(v) for k, v in EXPECTED_FILES.items()},
    "audit_csv_path": str(audit_csv_path),
}

save_json(duckdb_setup_metadata, METADATA_DIR / "duckdb_setup.json")

# ------------------------------
# 8) Output for feedback
# ------------------------------
print("\nDuckDB view registry summary:")
display(audit_duckdb_views[["view_name", "n_rows", "n_cols"]])

print("\nSaved:")
print("-", audit_csv_path.resolve())
print("-", (METADATA_DIR / "duckdb_setup.json").resolve())

print("\nNotes:")
print("- DuckDB connection object available as: con")
print("- Materialized tables will be stored later in DuckDB and/or outputs_benchmark_survival/data/")


A2 — DuckDB Setup and OULAD Registration
DuckDB file will be created/used at: /workspaces/survival_dropout_benchmark/outputs_benchmark_survival/benchmark_survival.duckdb
All expected OULAD CSV files are available.

Registered DuckDB views:
 - studentInfo
 - studentRegistration
 - studentVle
 - courses
 - vle
 - studentAssessment
 - assessments

DuckDB view registry summary:


,view_name,n_rows,n_cols
0,assessments,206,6
1,courses,22,3
2,studentAssessment,173912,5
3,studentInfo,32593,12
4,studentRegistration,32593,5
5,studentVle,10655280,6
6,vle,6364,6



Saved:
- /workspaces/survival_dropout_benchmark/outputs_benchmark_survival/tables/table_duckdb_registered_views.csv
- /workspaces/survival_dropout_benchmark/outputs_benchmark_survival/metadata/duckdb_setup.json

Notes:
- DuckDB connection object available as: con
- Materialized tables will be stored later in DuckDB and/or outputs_benchmark_survival/data/


## A3 — Backbone-Oriented Raw Audit with DuckDB


### Funcao: get_duckdb_columns

Definicao isolada de get_duckdb_columns para reutilizacao nas etapas seguintes.


In [19]:
# ==============================================================
# A3 — Backbone-Oriented Raw Audit with DuckDB
# --------------------------------------------------------------
# Purpose:
#   Audit the raw OULAD tables with a backbone-oriented focus,
#   validating whether the minimal enrollment backbone can be
#   built cleanly from studentInfo + studentRegistration, and
#   whether the transactional tables are structurally ready for
#   later temporal expansion.
#
# Inputs expected from previous cells:
#   - con (DuckDB connection)
#   - TABLES_DIR
#   - OUTPUT_DIR
#
# Outputs:
#   - table_oulad_backbone_oriented_audit.csv
#   - table_oulad_critical_missing_audit.csv
#   - table_oulad_backbone_key_uniqueness.csv
# ==============================================================

print("\n" + "=" * 70)
print("A3 — Backbone-Oriented Raw Audit with DuckDB")
print("=" * 70)

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = ["con", "TABLES_DIR", "OUTPUT_DIR"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

KEYS = ["id_student", "code_module", "code_presentation"]

# ------------------------------
# 2) Target tables and critical columns
# ------------------------------
audit_tables = {
    "studentInfo": {
        "role": "backbone_core",
        "critical_columns": ["id_student", "code_module", "code_presentation", "final_result"]
    },
    "studentRegistration": {
        "role": "backbone_core",
        "critical_columns": ["id_student", "code_module", "code_presentation", "date_registration", "date_unregistration"]
    },
    "studentVle": {
        "role": "transactional_minimal",
        "critical_columns": ["id_student", "code_module", "code_presentation", "id_site", "date", "sum_click"]
    },
    "studentAssessment": {
        "role": "transactional_optional",
        "critical_columns": ["id_student", "code_module", "code_presentation", "id_assessment", "date_submitted"]
    },
    "assessments": {
        "role": "transactional_optional",
        "critical_columns": ["id_assessment", "code_module", "code_presentation"]
    },
    "courses": {
        "role": "auxiliary",
        "critical_columns": ["code_module", "code_presentation"]
    },
    "vle": {
        "role": "auxiliary",
        "critical_columns": ["id_site", "code_module", "code_presentation"]
    },
}

# ------------------------------
# 3) Helper: get column names from DuckDB
# ------------------------------
def get_duckdb_columns(view_name: str) -> list:
    desc_df = con.execute(f"DESCRIBE {view_name}").fetchdf()
    return desc_df["column_name"].tolist()

# ------------------------------
# 4) Structural audit
# ------------------------------
structural_rows = []

for table_name, meta in audit_tables.items():
    columns = get_duckdb_columns(table_name)
    n_rows = con.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
    n_cols = len(columns)

    critical_cols = meta["critical_columns"]
    present_critical = [c for c in critical_cols if c in columns]
    missing_critical = [c for c in critical_cols if c not in columns]

    structural_rows.append({
        "table_name": table_name,
        "role": meta["role"],
        "n_rows": int(n_rows),
        "n_cols": int(n_cols),
        "critical_columns_expected": ", ".join(critical_cols),
        "critical_columns_present": ", ".join(present_critical),
        "critical_columns_missing": ", ".join(missing_critical),
        "all_columns": ", ".join(columns)
    })

audit_structural = pd.DataFrame(structural_rows).sort_values(["role", "table_name"]).reset_index(drop=True)

# ------------------------------
# 5) Critical missing audit
# ------------------------------
missing_rows = []

for table_name, meta in audit_tables.items():
    columns = get_duckdb_columns(table_name)

    for col in meta["critical_columns"]:
        if col in columns:
            q = f"""
            SELECT
                COUNT(*) AS n_total,
                SUM(CASE WHEN {col} IS NULL THEN 1 ELSE 0 END) AS n_missing
            FROM {table_name}
            """
            result = con.execute(q).fetchdf().iloc[0]

            n_total = int(result["n_total"])
            n_missing = int(result["n_missing"]) if pd.notna(result["n_missing"]) else 0
            pct_missing = (n_missing / n_total * 100.0) if n_total > 0 else np.nan
        else:
            n_total = None
            n_missing = None
            pct_missing = None

        missing_rows.append({
            "table_name": table_name,
            "column_name": col,
            "exists_in_table": col in columns,
            "n_total": n_total,
            "n_missing": n_missing,
            "pct_missing": pct_missing
        })

audit_missing = pd.DataFrame(missing_rows).sort_values(["table_name", "column_name"]).reset_index(drop=True)

# ------------------------------
# 6) Backbone key uniqueness audit
# ------------------------------
uniqueness_rows = []

for table_name in ["studentInfo", "studentRegistration"]:
    total_rows = con.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]

    distinct_rows = con.execute(f"""
        SELECT COUNT(*) 
        FROM (
            SELECT DISTINCT id_student, code_module, code_presentation
            FROM {table_name}
        ) t
    """).fetchone()[0]

    duplicate_rows = int(total_rows - distinct_rows)

    uniqueness_rows.append({
        "table_name": table_name,
        "keys_checked": ", ".join(KEYS),
        "n_rows_total": int(total_rows),
        "n_distinct_keys": int(distinct_rows),
        "n_excess_duplicate_rows": duplicate_rows,
        "is_unique_by_keys": duplicate_rows == 0
    })

audit_uniqueness = pd.DataFrame(uniqueness_rows)

# ------------------------------
# 7) Save outputs
# ------------------------------
structural_path = TABLES_DIR / "table_oulad_backbone_oriented_audit.csv"
missing_path = TABLES_DIR / "table_oulad_critical_missing_audit.csv"
uniqueness_path = TABLES_DIR / "table_oulad_backbone_key_uniqueness.csv"

audit_structural.to_csv(structural_path, index=False)
audit_missing.to_csv(missing_path, index=False)
audit_uniqueness.to_csv(uniqueness_path, index=False)

# ------------------------------
# 8) Output for feedback
# ------------------------------
print("\nBackbone-oriented structural summary:")
display(audit_structural[[
    "table_name", "role", "n_rows", "n_cols", "critical_columns_missing"
]])

print("\nCritical missing summary (only rows with missing > 0 or missing columns):")
audit_missing_filtered = audit_missing[
    (~audit_missing["exists_in_table"]) | (audit_missing["n_missing"].fillna(0) > 0)
].copy()
display(audit_missing_filtered)

print("\nBackbone key uniqueness:")
display(audit_uniqueness)

print("\nSaved:")
print("-", structural_path.resolve())
print("-", missing_path.resolve())
print("-", uniqueness_path.resolve())


A3 — Backbone-Oriented Raw Audit with DuckDB

Backbone-oriented structural summary:


,table_name,role,n_rows,n_cols,critical_columns_missing
0,courses,auxiliary,22,3,
1,vle,auxiliary,6364,6,
2,studentInfo,backbone_core,32593,12,
3,studentRegistration,backbone_core,32593,5,
4,studentVle,transactional_minimal,10655280,6,
5,assessments,transactional_optional,206,6,
6,studentAssessment,transactional_optional,173912,5,"code_module, code_presentation"



Critical missing summary (only rows with missing > 0 or missing columns):


,table_name,column_name,exists_in_table,n_total,n_missing,pct_missing
5,studentAssessment,code_module,False,NaN,NaN,NaN
6,studentAssessment,code_presentation,False,NaN,NaN,NaN
16,studentRegistration,date_registration,True,32593.0,45.0,0.138066
17,studentRegistration,date_unregistration,True,32593.0,22521.0,69.097659



Backbone key uniqueness:


,table_name,keys_checked,n_rows_total,n_distinct_keys,n_excess_duplicate_rows,is_unique_by_keys
0,studentInfo,"id_student, code_module, code_presentation",32593,32593,0,True
1,studentRegistration,"id_student, code_module, code_presentation",32593,32593,0,True



Saved:
- /workspaces/survival_dropout_benchmark/outputs_benchmark_survival/tables/table_oulad_backbone_oriented_audit.csv
- /workspaces/survival_dropout_benchmark/outputs_benchmark_survival/tables/table_oulad_critical_missing_audit.csv
- /workspaces/survival_dropout_benchmark/outputs_benchmark_survival/tables/table_oulad_backbone_key_uniqueness.csv


## A4 — Build Minimal Enrollment Backbone


In [20]:
# ==============================================================
# A4 — Build Minimal Enrollment Backbone
# --------------------------------------------------------------
# Purpose:
#   Materialize the minimal enrollment-level backbone table
#   by joining studentInfo and studentRegistration on the
#   enrollment key, preserving one row per enrollment and
#   exporting the result for downstream survival preparation.
#
# Inputs expected from previous cells:
#   - con
#   - OUTPUT_DIR
#   - TABLES_DIR
#
# Outputs:
#   - DuckDB table: enrollment_backbone
#   - enrollment_backbone.parquet
#   - enrollment_backbone.csv
#   - table_enrollment_backbone_audit.csv
# ==============================================================

print("\n" + "=" * 70)
print("A4 — Build Minimal Enrollment Backbone")
print("=" * 70)

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = ["con", "OUTPUT_DIR", "TABLES_DIR"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

DATA_OUTPUT_DIR = OUTPUT_DIR / "data"
DATA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

KEYS = ["id_student", "code_module", "code_presentation"]

# ------------------------------
# 2) Create minimal enrollment backbone
# ------------------------------
con.execute("DROP TABLE IF EXISTS enrollment_backbone")

create_sql = """
CREATE TABLE enrollment_backbone AS
SELECT
    si.id_student,
    si.code_module,
    si.code_presentation,
    si.gender,
    si.region,
    si.highest_education,
    si.imd_band,
    si.age_band,
    si.num_of_prev_attempts,
    si.studied_credits,
    si.disability,
    si.final_result,
    sr.date_registration,
    sr.date_unregistration
FROM studentInfo AS si
LEFT JOIN studentRegistration AS sr
    ON si.id_student = sr.id_student
   AND si.code_module = sr.code_module
   AND si.code_presentation = sr.code_presentation
"""

con.execute(create_sql)

# ------------------------------
# 3) Validate uniqueness after join
# ------------------------------
n_rows_total = con.execute("SELECT COUNT(*) FROM enrollment_backbone").fetchone()[0]

n_distinct_keys = con.execute("""
    SELECT COUNT(*)
    FROM (
        SELECT DISTINCT id_student, code_module, code_presentation
        FROM enrollment_backbone
    ) t
""").fetchone()[0]

is_unique = (n_rows_total == n_distinct_keys)

# ------------------------------
# 4) Final result distribution
# ------------------------------
final_result_dist = con.execute("""
    SELECT
        final_result,
        COUNT(*) AS n
    FROM enrollment_backbone
    GROUP BY final_result
    ORDER BY n DESC
""").fetchdf()

final_result_dist["pct"] = final_result_dist["n"] / final_result_dist["n"].sum() * 100.0

# ------------------------------
# 5) Missing audit for key date fields
# ------------------------------
date_missing_audit = con.execute("""
    SELECT
        COUNT(*) AS n_total,
        SUM(CASE WHEN date_registration IS NULL THEN 1 ELSE 0 END) AS n_missing_date_registration,
        SUM(CASE WHEN date_unregistration IS NULL THEN 1 ELSE 0 END) AS n_missing_date_unregistration
    FROM enrollment_backbone
""").fetchdf()

date_missing_audit["pct_missing_date_registration"] = (
    date_missing_audit["n_missing_date_registration"] / date_missing_audit["n_total"] * 100.0
)
date_missing_audit["pct_missing_date_unregistration"] = (
    date_missing_audit["n_missing_date_unregistration"] / date_missing_audit["n_total"] * 100.0
)

# ------------------------------
# 6) Build audit table
# ------------------------------
audit_rows = [{
    "table_name": "enrollment_backbone",
    "n_rows_total": int(n_rows_total),
    "n_distinct_enrollments": int(n_distinct_keys),
    "is_unique_by_keys": bool(is_unique),
    "n_missing_date_registration": int(date_missing_audit.loc[0, "n_missing_date_registration"]),
    "pct_missing_date_registration": float(date_missing_audit.loc[0, "pct_missing_date_registration"]),
    "n_missing_date_unregistration": int(date_missing_audit.loc[0, "n_missing_date_unregistration"]),
    "pct_missing_date_unregistration": float(date_missing_audit.loc[0, "pct_missing_date_unregistration"]),
}]

audit_backbone = pd.DataFrame(audit_rows)

# ------------------------------
# 7) Export backbone
# ------------------------------
parquet_path = DATA_OUTPUT_DIR / "enrollment_backbone.parquet"
csv_path = DATA_OUTPUT_DIR / "enrollment_backbone.csv"
audit_path = TABLES_DIR / "table_enrollment_backbone_audit.csv"

con.execute(f"COPY enrollment_backbone TO '{parquet_path.as_posix()}' (FORMAT PARQUET)")
con.execute(f"COPY enrollment_backbone TO '{csv_path.as_posix()}' (HEADER, DELIMITER ',')")
audit_backbone.to_csv(audit_path, index=False)

# ------------------------------
# 8) Output for feedback
# ------------------------------
print("\nBackbone table summary:")
display(audit_backbone)

print("\nFinal result distribution:")
display(final_result_dist)

print("\nSaved:")
print("-", parquet_path.resolve())
print("-", csv_path.resolve())
print("-", audit_path.resolve())


A4 — Build Minimal Enrollment Backbone

Backbone table summary:


,table_name,n_rows_total,n_distinct_enrollments,is_unique_by_keys,n_missing_date_registration,pct_missing_date_registration,n_missing_date_unregistration,pct_missing_date_unregistration
0,enrollment_backbone,32593,32593,True,45,0.138066,22521,69.097659



Final result distribution:


,final_result,n,pct
0,Pass,12361,37.925321
1,Withdrawn,10156,31.160065
2,Fail,7052,21.636548
3,Distinction,3024,9.278066



Saved:
- /workspaces/survival_dropout_benchmark/outputs_benchmark_survival/data/enrollment_backbone.parquet
- /workspaces/survival_dropout_benchmark/outputs_benchmark_survival/data/enrollment_backbone.csv
- /workspaces/survival_dropout_benchmark/outputs_benchmark_survival/tables/table_enrollment_backbone_audit.csv


## A5 — Final Event, Time, and Censoring Logic


In [21]:
# ==============================================================
# A5 — Final event/time construction for enrollment_survival_ready
# --------------------------------------------------------------
# Purpose:
#   Build the canonical enrollment-level survival-ready table
#   from the minimal enrollment backbone plus enrollment-level
#   VLE activity aggregates, then derive the final event/time
#   variables for temporal survival modeling.
#
# Inputs expected from previous cells:
#   - con
#   - OUTPUT_DIR
#   - TABLES_DIR
#   - enrollment_backbone (DuckDB table from A4)
#   - studentVle (DuckDB table/view loaded upstream)
#
# Outputs:
#   - DuckDB table: enrollment_survival_ready
#   - enrollment_survival_ready.parquet
#   - enrollment_survival_ready.csv
#   - table_enrollment_survival_ready_audit.csv
#   - table_enrollment_survival_ready_final_result_by_event.csv
# ==============================================================

print("\n" + "=" * 70)
print("A5 — Final event/time construction for enrollment_survival_ready")
print("=" * 70)

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = ["con", "OUTPUT_DIR", "TABLES_DIR"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

DATA_OUTPUT_DIR = OUTPUT_DIR / "data"
DATA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

available_tables = set(
    con.execute("SHOW TABLES").fetchdf()["name"].astype(str).tolist()
)

required_tables = ["enrollment_backbone", "studentVle"]
missing_tables = [t for t in required_tables if t not in available_tables]
if missing_tables:
    raise RuntimeError(
        "Missing required DuckDB table(s)/view(s): "
        + ", ".join(missing_tables)
        + ". Please run the upstream preparation cells first."
    )

# ------------------------------
# 2) Aggregate VLE activity to enrollment level
# ------------------------------
con.execute("DROP TABLE IF EXISTS tmp_p5_vle_enrollment_agg")

con.execute("""
CREATE TABLE tmp_p5_vle_enrollment_agg AS
SELECT
    id_student,
    code_module,
    code_presentation,
    MAX(date) AS max_vle_day,
    COUNT(*) AS n_vle_rows,
    COALESCE(SUM(sum_click), 0) AS total_clicks_all_time,
    CASE WHEN COUNT(*) > 0 THEN 1 ELSE 0 END AS has_any_vle_activity
FROM studentVle
GROUP BY
    id_student,
    code_module,
    code_presentation
""")

# ------------------------------
# 3) Build raw survival-ready base
# ------------------------------
con.execute("DROP TABLE IF EXISTS enrollment_survival_ready_raw")

con.execute("""
CREATE TABLE enrollment_survival_ready_raw AS
SELECT
    eb.*,

    COALESCE(v.has_any_vle_activity, 0) AS has_any_vle_activity,
    v.max_vle_day,
    COALESCE(v.n_vle_rows, 0) AS n_vle_rows,
    COALESCE(v.total_clicks_all_time, 0) AS total_clicks_all_time

FROM enrollment_backbone AS eb
LEFT JOIN tmp_p5_vle_enrollment_agg AS v
    ON eb.id_student = v.id_student
   AND eb.code_module = v.code_module
   AND eb.code_presentation = v.code_presentation
""")

base = con.execute("""
SELECT *
FROM enrollment_survival_ready_raw
""").fetchdf()

# ------------------------------
# 4) Final event/time logic in pandas
# ------------------------------
import numpy as np
import pandas as pd

df = base.copy()

# Canonical enrollment identifier
df["enrollment_id"] = (
    df["id_student"].astype(str)
    + "||" + df["code_module"].astype(str)
    + "||" + df["code_presentation"].astype(str)
)

# Robust coercions
date_registration_num = pd.to_numeric(df["date_registration"], errors="coerce")
date_unregistration_num = pd.to_numeric(df["date_unregistration"], errors="coerce")
max_vle_day_num = pd.to_numeric(df["max_vle_day"], errors="coerce")
n_vle_rows_num = pd.to_numeric(df["n_vle_rows"], errors="coerce").fillna(0)
total_clicks_all_time_num = pd.to_numeric(df["total_clicks_all_time"], errors="coerce").fillna(0)
has_any_vle_activity_num = pd.to_numeric(df["has_any_vle_activity"], errors="coerce").fillna(0).astype(int)

# Withdrawn and event observability
final_result_clean = df["final_result"].astype(str).str.strip().str.upper()

df["is_withdrawn"] = (final_result_clean == "WITHDRAWN").astype(int)
df["has_valid_unregistration_date"] = (
    date_unregistration_num.notna() & (date_unregistration_num >= 0)
).astype(int)

# Event observed = withdrawn with valid observed unregistration date
df["event_observed"] = (
    (df["is_withdrawn"] == 1) &
    (df["has_valid_unregistration_date"] == 1)
).astype(int)

# Withdrawn without usable event date
df["is_withdrawn_without_valid_unregistration"] = (
    (df["is_withdrawn"] == 1) &
    (df["event_observed"] == 0)
).astype(int)

# Event week
df["t_event_week"] = np.where(
    df["event_observed"] == 1,
    np.floor(date_unregistration_num / 7.0),
    np.nan
)

# Raw censoring week based on latest observed VLE activity
df["t_last_obs_week_raw"] = np.where(
    max_vle_day_num.notna(),
    np.floor(max_vle_day_num / 7.0),
    np.nan
)

# Flag cases with only pre-start activity
df["has_pre_start_only_activity"] = (
    (has_any_vle_activity_num == 1) &
    max_vle_day_num.notna() &
    (max_vle_day_num < 0)
).astype(int)

# Final censoring week:
# - if no activity at all -> fallback to 0
# - if activity only before official start -> clamp to 0
# - otherwise -> floor(max_vle_day / 7), clamped at 0
df["t_last_obs_week"] = np.where(
    has_any_vle_activity_num == 0,
    0,
    np.where(
        max_vle_day_num.notna(),
        np.maximum(np.floor(max_vle_day_num / 7.0), 0),
        0
    )
)

# Final analysis time:
# - event week if event observed
# - censoring week otherwise
df["t_final_week"] = np.where(
    df["event_observed"] == 1,
    np.floor(date_unregistration_num / 7.0),
    df["t_last_obs_week"]
)

# Clamp final times to valid non-negative range
df["t_event_week"] = np.where(
    pd.notna(df["t_event_week"]),
    np.maximum(df["t_event_week"], 0),
    np.nan
)
df["t_last_obs_week"] = np.maximum(df["t_last_obs_week"], 0)
df["t_final_week"] = np.maximum(df["t_final_week"], 0)

# Integer typing where appropriate
df["t_last_obs_week"] = pd.to_numeric(df["t_last_obs_week"], errors="coerce").fillna(0).astype(int)
df["t_final_week"] = pd.to_numeric(df["t_final_week"], errors="coerce").fillna(0).astype(int)

# Audit flags
df["used_zero_week_fallback_for_censoring"] = (
    (df["event_observed"] == 0) &
    (
        (has_any_vle_activity_num == 0) |
        (df["has_pre_start_only_activity"] == 1)
    )
).astype(int)

df["t_last_obs_week_was_clamped"] = (
    pd.notna(df["t_last_obs_week_raw"]) &
    (pd.to_numeric(df["t_last_obs_week_raw"], errors="coerce") < 0)
).astype(int)

df["t_final_week_was_clamped"] = (
    (df["event_observed"] == 0) &
    (df["t_last_obs_week_was_clamped"] == 1)
).astype(int)

# Reorder some important columns near the front
priority_cols = [
    "enrollment_id",
    "id_student", "code_module", "code_presentation",
    "final_result",
    "date_registration", "date_unregistration",
    "is_withdrawn", "has_valid_unregistration_date",
    "event_observed", "is_withdrawn_without_valid_unregistration",
    "has_any_vle_activity", "max_vle_day", "n_vle_rows", "total_clicks_all_time",
    "t_event_week", "t_last_obs_week_raw", "t_last_obs_week", "t_final_week",
    "has_pre_start_only_activity",
    "used_zero_week_fallback_for_censoring",
    "t_last_obs_week_was_clamped",
    "t_final_week_was_clamped"
]
remaining_cols = [c for c in df.columns if c not in priority_cols]
df = df[priority_cols + remaining_cols]

# ------------------------------
# 5) Materialize final DuckDB table
# ------------------------------
con.execute("DROP TABLE IF EXISTS enrollment_survival_ready")
con.register("enrollment_survival_ready_df", df)
con.execute("""
CREATE TABLE enrollment_survival_ready AS
SELECT *
FROM enrollment_survival_ready_df
""")
con.unregister("enrollment_survival_ready_df")

# ------------------------------
# 6) Audits
# ------------------------------
n_total = len(df)
n_unique_enrollments = df["enrollment_id"].nunique()
is_unique_by_enrollment = (n_total == n_unique_enrollments)

n_withdrawn = int(df["is_withdrawn"].sum())
n_event_observed = int(df["event_observed"].sum())
n_withdrawn_without_valid_unregistration = int(df["is_withdrawn_without_valid_unregistration"].sum())
n_no_vle_activity = int((df["has_any_vle_activity"] == 0).sum())
n_pre_start_only = int(df["has_pre_start_only_activity"].sum())
n_zero_week_fallback = int(df["used_zero_week_fallback_for_censoring"].sum())

max_t_event_week = (
    float(pd.to_numeric(df["t_event_week"], errors="coerce").dropna().max())
    if pd.to_numeric(df["t_event_week"], errors="coerce").notna().any()
    else np.nan
)
max_t_last_obs_week = float(pd.to_numeric(df["t_last_obs_week"], errors="coerce").max())
max_t_final_week = float(pd.to_numeric(df["t_final_week"], errors="coerce").max())

audit_rows = [{
    "table_name": "enrollment_survival_ready",
    "n_rows_total": int(n_total),
    "n_unique_enrollments": int(n_unique_enrollments),
    "is_unique_by_enrollment": bool(is_unique_by_enrollment),
    "n_withdrawn": n_withdrawn,
    "n_event_observed": n_event_observed,
    "event_rate_over_all_enrollments": float(n_event_observed / n_total) if n_total > 0 else np.nan,
    "n_withdrawn_without_valid_unregistration": n_withdrawn_without_valid_unregistration,
    "n_no_vle_activity": n_no_vle_activity,
    "n_pre_start_only_activity": n_pre_start_only,
    "n_zero_week_fallback_for_censoring": n_zero_week_fallback,
    "max_t_event_week": max_t_event_week,
    "max_t_last_obs_week": max_t_last_obs_week,
    "max_t_final_week": max_t_final_week
}]
audit_df = pd.DataFrame(audit_rows)

final_result_by_event = (
    df.groupby(["final_result", "event_observed"], dropna=False)
      .size()
      .reset_index(name="n")
      .sort_values(["final_result", "event_observed"], ascending=[True, True])
)

final_result_by_event["pct_within_final_result"] = (
    final_result_by_event["n"] /
    final_result_by_event.groupby("final_result")["n"].transform("sum") * 100.0
)

# ------------------------------
# 7) Exports
# ------------------------------
parquet_path = DATA_OUTPUT_DIR / "enrollment_survival_ready.parquet"
csv_path = DATA_OUTPUT_DIR / "enrollment_survival_ready.csv"
audit_path = TABLES_DIR / "table_enrollment_survival_ready_audit.csv"
result_event_path = TABLES_DIR / "table_enrollment_survival_ready_final_result_by_event.csv"

con.execute(f"COPY enrollment_survival_ready TO '{parquet_path.as_posix()}' (FORMAT PARQUET)")
con.execute(f"COPY enrollment_survival_ready TO '{csv_path.as_posix()}' (HEADER, DELIMITER ',')")

audit_df.to_csv(audit_path, index=False)
final_result_by_event.to_csv(result_event_path, index=False)

# ------------------------------
# 8) Output for feedback
# ------------------------------
print("\nEnrollment survival-ready audit:")
display(audit_df)

print("\nFinal result × event observed:")
display(final_result_by_event)

print("\nSaved:")
print("-", parquet_path.resolve())
print("-", csv_path.resolve())
print("-", audit_path.resolve())
print("-", result_event_path.resolve())


A5 — Final event/time construction for enrollment_survival_ready

Enrollment survival-ready audit:


,table_name,n_rows_total,n_unique_enrollments,is_unique_by_enrollment,n_withdrawn,n_event_observed,event_rate_over_all_enrollments,n_withdrawn_without_valid_unregistration,n_no_vle_activity,n_pre_start_only_activity,n_zero_week_fallback_for_censoring,max_t_event_week,max_t_last_obs_week,max_t_final_week
0,enrollment_survival_ready,32593,32593,True,10156,7387,0.226644,2769,3365,728,3078,63.0,38.0,63.0



Final result × event observed:


,final_result,event_observed,n,pct_within_final_result
0,Distinction,0,3024,100.000000
1,Fail,0,7052,100.000000
2,Pass,0,12361,100.000000
3,Withdrawn,0,2769,27.264671
4,Withdrawn,1,7387,72.735329



Saved:
- /workspaces/survival_dropout_benchmark/outputs_benchmark_survival/data/enrollment_survival_ready.parquet
- /workspaces/survival_dropout_benchmark/outputs_benchmark_survival/data/enrollment_survival_ready.csv
- /workspaces/survival_dropout_benchmark/outputs_benchmark_survival/tables/table_enrollment_survival_ready_audit.csv
- /workspaces/survival_dropout_benchmark/outputs_benchmark_survival/tables/table_enrollment_survival_ready_final_result_by_event.csv
